# Cross-Validation — How to Score a Model Reliably

In every notebook so far we judged a model on a **single train/test split**. The problem: that score depends on *which* rows happened to land in the test set — change the split and the number moves. So a single accuracy value can be misleading.

**Cross-validation** fixes this: instead of trusting one split, we test the model on **many** splits and average the results — giving a far more reliable score, plus a sense of how much it varies.

### What we'll cover
1. **The problem** — how much a single split's score jumps around.
2. **K-Fold cross-validation** — split the data into *k* folds, let each be the test set once, then average. We'll build it by hand first.
3. **`cross_val_score`** — the one-line shortcut for that whole loop.
4. **StratifiedKFold** — the classification-friendly version that keeps class balance in every fold.

**Dataset:** the Heart Disease data (`heart_disease.csv`) — predict whether a patient has heart disease (`target = 1`) or not (`target = 0`).

## Load and prepare the data

Read the CSV, treat `'?'` as missing and drop those rows, then build a binary `target` column.

In [1]:
import numpy as np
import pandas as pd

# Load + quick clean: '?' -> NaN, drop them, then make a binary target
df = pd.read_csv('heart_disease.csv', na_values='?').dropna()
df['target'] = (df['num'] > 0).astype(int)   # 1 = disease, 0 = healthy

X = df.drop(columns=['num', 'target'])   # 13 clinical features
y = df['target']                         # what we want to predict
print('rows:', len(X), ' features:', X.shape[1])

rows: 297  features: 13


## 1. The problem: one train/test split is unreliable

Let's train two models — an **SVM** and a **Random Forest** — on a single 80/20 split and look at their test scores. Keep an eye on the numbers: if you change `random_state` in the split below, they jump around by several percentage points. That instability is exactly what cross-validation will fix.

In [2]:
from sklearn.model_selection import train_test_split

# a single 80/20 split -- try changing random_state and watch the scores move
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

### SVM

In [3]:
from sklearn.svm import SVC

svm = SVC()
svm.fit(X_train, y_train)
svm.score(X_test, y_test)   # accuracy on this one test split

0.6666666666666666

### Random Forest

In [4]:
from sklearn.ensemble import RandomForestClassifier

# random_state=42 keeps the forest reproducible (it's random without it)
rf = RandomForestClassifier(n_estimators=40, random_state=42)
rf.fit(X_train, y_train)
rf.score(X_test, y_test)   # accuracy on this one test split

0.85

Both scores above came from **one** split. Rerun the split with a different `random_state` and they can swing a lot — so a single number isn't trustworthy. The fix is to test on **many** splits and average them. That's cross-validation.

## 2. K-Fold Cross Validation

**K-Fold** splits the data into *k* equal parts ("folds"). It then runs *k* rounds — each round, one fold is the **test** set and the other *k−1* folds are the **training** set. Every row is tested exactly once, and we average the *k* scores.

First, create a `KFold` splitter with 3 folds:

In [5]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=3)
kf

KFold(n_splits=3, random_state=None, shuffle=False)

To see how it works, here's `KFold` splitting a simple list of 9 numbers into 3 folds. Each line prints the **train indices** and then the **test indices** for that round:

In [6]:
data = [1, 2, 3, 4, 5, 6, 7, 8, 9]

for train_index, test_index in kf.split(data):
    print(train_index, test_index)

[3 4 5 6 7 8] [0 1 2]
[0 1 2 6 7 8] [3 4 5]
[0 1 2 3 4 5] [6 7 8]


Notice the test indices march through the data in **contiguous blocks** — `[0 1 2]`, then `[3 4 5]`, then `[6 7 8]`. (`KFold` doesn't shuffle by default.)

Now let's use those folds for real: train the SVM and Random Forest on each fold and collect the scores.

In [7]:
# Manually run 3-fold cross-validation for both models
svm_scores = []
rf_scores = []

for train_index, test_index in kf.split(X, y):
    # use .iloc for positional row selection (X is a DataFrame, so plain X[idx] would look for columns)
    X_train, X_test, y_train, y_test = X.iloc[train_index], X.iloc[test_index], y.iloc[train_index], y.iloc[test_index]

    svm = SVC()
    svm.fit(X_train, y_train)
    svm_scores.append(svm.score(X_test, y_test))

    rf = RandomForestClassifier(n_estimators=40, random_state=42)
    rf.fit(X_train, y_train)
    rf_scores.append(rf.score(X_test, y_test))

In [8]:
svm_scores   # one score per fold

[0.6161616161616161, 0.6565656565656566, 0.6060606060606061]

In [9]:
print("Average SVM score: ", np.mean(svm_scores))

Average SVM score:  0.6262626262626263


In [10]:
rf_scores   # one score per fold

[0.8383838383838383, 0.8080808080808081, 0.797979797979798]

In [11]:
print("Average Random Forest score: ", np.mean(rf_scores))

Average Random Forest score:  0.8148148148148149


The average is a single, trustworthy score for each model — and the Random Forest (~0.81) clearly beats the SVM (~0.63) across all folds, not just on one lucky split.

## 3. `cross_val_score` — the shortcut

That manual loop is so common that scikit-learn wraps it in a **single line**. You pass the model, `X`, `y`, and a `cv` strategy, and it returns the array of per-fold scores. Handing it our **same `kf`** reproduces the manual numbers exactly.

In [12]:
from sklearn.model_selection import cross_val_score

In [13]:
# SVM -- same as the manual loop, in one line
cross_val_score(SVC(), X, y, cv=kf)

array([0.61616162, 0.65656566, 0.60606061])

In [14]:
# Random Forest -- random_state=42 keeps it reproducible
cross_val_score(RandomForestClassifier(n_estimators=40, random_state=42), X, y, cv=kf)

array([0.83838384, 0.80808081, 0.7979798 ])

These match our manual loop exactly — `cross_val_score` is simply doing that loop for us.

> **Tip:** in everyday use you just pass an integer, e.g. `cross_val_score(model, X, y, cv=3)`, and let scikit-learn build the folds for you.

## 4. Stratified K Fold

`StratifiedKFold` works just like `KFold`, but it keeps the **same class balance** in every fold. Because it needs the class labels to do that, you must pass **both** `X` and `y` to `.split()` (passing only the data is what triggers a `missing argument 'y'` error).

This matters for classification: with plain `KFold`, a fold could accidentally get too few of one class and give a misleading score.

In [15]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=3)

# same idea as the KFold demo -- but StratifiedKFold also needs the labels (y)
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9]
labels  = [0, 0, 0, 0, 0, 1, 1, 1, 1]   # 5 of class 0, 4 of class 1

for train_index, test_index in skf.split(numbers, labels):
    print(train_index, test_index)

[2 3 4 6 7 8] [0 1 5]
[0 1 4 5 7 8] [2 3 6]
[0 1 2 3 5 6] [4 7 8]


See how each fold's **test set mixes both classes** (about the same 5:4 ratio as the full list) instead of the contiguous blocks plain `KFold` gave. That's stratification keeping the balance.

Now the same one-liner as before, but passing our `StratifiedKFold`:

In [16]:
# SVM with StratifiedKFold
cross_val_score(SVC(), X, y, cv=skf)

array([0.58585859, 0.66666667, 0.60606061])

In [17]:
# Random Forest with StratifiedKFold
cross_val_score(RandomForestClassifier(n_estimators=40, random_state=42), X, y, cv=skf)

array([0.84848485, 0.7979798 , 0.78787879])

These scores are **identical** to `cross_val_score(model, X, y, cv=3)` — because for a classifier, passing an integer `cv=3` already uses `StratifiedKFold` under the hood. So you rarely need to build one by hand; it's just good to know that's what `cv=3` is doing.

## Recap

- **A single train/test split is noisy** — its score depends on which rows land in the test set, so it can swing several points just by changing `random_state`.
- **K-Fold cross-validation** splits the data into *k* folds, tests on each fold once, and **averages** — a far more reliable score, and the spread across folds tells you how much it varies.
- **`cross_val_score(model, X, y, cv=...)`** does that whole loop in one line.
- **`StratifiedKFold`** keeps each fold's class balance the same as the full dataset — the right default for classification. `cross_val_score` uses it automatically when you pass an integer `cv` with a classifier.
- Set **`random_state`** on random models (like Random Forest) when you want repeatable numbers.

**In practice**, you'll rarely write the manual loop — you'll just call `cross_val_score(model, X, y, cv=5)` to compare models or settings on a trustworthy averaged score. Everything above is what that one line is doing under the hood.

**Try next:** `GridSearchCV`, which uses cross-validation to search many hyperparameter combinations automatically and hand you the best one.